# CorpusHop — Bridge Pipeline

This notebook is a reusable version of the **3-hop Bridge** pipeline.

### What you need to change
Usually, only edit the **Configuration** cell below:

- `INPUT_DATA`: a JSON/JSONL file **or a directory** containing JSON/JSONL files.
- `FIELD_MAP`: only needed when your field names are unusual.
- `OUTPUT_DIR`: where cleaned data, graph files, caches, and generated questions are written.
- model / generation settings.

### Minimum input requirements
Each chunk needs:

1. a text field, e.g. `text`, `content`, or `context`
2. a triple field containing `[subject, relation, object]` triples, e.g. `extracts` or `triples`

Chunk IDs are optional; if absent, stable IDs are generated. `entities` and `topic` are optional.

The loader automatically recognizes several common field names and converts all data to the canonical schema used internally:

```json
{
  "id": "chunk_001",
  "topic": "optional-topic",
  "text": "chunk text",
  "extracts": [["subject", "relation", "object"]],
  "entities": ["subject", "object"]
}
```

> The reasoning logic is unchanged: exact normalized object→subject links, greedy lowest degree-sum path selection, node-disjoint acceptance on PASS, and edge-only removal on FAIL.


## Dependencies and API key

Install the required packages once:

```bash
pip install openai python-dotenv networkx numpy scipy
```

Set your OpenAI key without committing it to GitHub:

```bash
# .env
OPENAI_API_KEY=your_key_here
```

Recommended `.gitignore` entries:

```text
.env
outputs/
__pycache__/
.ipynb_checkpoints/
```

For a custom dataset, you should only need to change `INPUT_DATA` and, if auto-detection cannot recognize your column names, `FIELD_MAP`.


In [ ]:
from pathlib import Path

# =============================================================================
# CONFIGURATION — normally this is the only cell you need to edit
# =============================================================================

# Input may be:
#   1) one .json / .jsonl / .ndjson file, OR
#   2) a directory containing those files.
INPUT_DATA = Path("./data/chunks.jsonl")

# All generated files are written here.
OUTPUT_DIR = Path("./outputs")

# Optional field mapping.
# Use None to auto-detect common aliases.
# Example for a custom dataset:
# FIELD_MAP = {
#     "id": "chunk_key",
#     "text": "passage",
#     "triples": "spo_list",
#     "entities": None,
#     "topic": "category",
# }
FIELD_MAP = {
    "id": None,
    "text": None,
    "triples": None,
    "entities": None,
    "topic": None,
}

# Search nested directories when INPUT_DATA is a directory.
RECURSIVE_INPUT = True

# Generated filenames. Usually you do not need to change these.
CLEANED_FILENAME = "docs_with_triples_cleaned.json"
GRAPH_FILENAME = "graph_data_exact.pkl"
CONNECTIONS_FILENAME = "node_connections_exact.txt"

# OpenAI settings.
# Put OPENAI_API_KEY in your environment or in a local .env file.
GENERATION_MODEL = "gpt-4o-mini"
JUDGE_MODEL_NAME = "gpt-4o-mini"
GENERATION_TEMPERATURE = 0.3

# None = continue until no valid path remains.
MAX_SELECTED_PATHS = None

# Performance settings.
CHAIN_DB_BATCH_SIZE = 2_000
PHASE2_SAVE_EVERY = 500

# Execution switches.
# Cleaning + graph building do not call an LLM.
RUN_CLEANING = True
RUN_GRAPH_BUILDING = True

# This stage calls the OpenAI API and can incur cost.
# Set to True when you are ready to generate and judge questions.
RUN_LLM_PIPELINE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input : {INPUT_DATA}")
print(f"Output: {OUTPUT_DIR.resolve()}")


## 1. Generic input loader and canonical schema

In [ ]:
import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Iterable


SUPPORTED_EXTENSIONS = {".json", ".jsonl", ".ndjson", ".txt"}

FIELD_ALIASES = {
    "id": (
        "id", "chunk_id", "chunkid", "doc_id", "document_id",
        "passage_id", "section_id", "uid",
    ),
    "text": (
        "text", "content", "context", "passage", "chunk_text",
        "document", "body",
    ),
    "triples": (
        "extracts", "triples", "relations", "spo", "spo_triples",
        "knowledge_triples",
    ),
    "entities": (
        "entities", "entity_list", "named_entities",
    ),
    "topic": (
        "topic", "topic_id", "category", "domain", "group",
    ),
}


def _pick_field(record: dict, canonical_name: str, field_map: dict) -> str | None:
    """Resolve one canonical field to an actual key in a record."""
    explicit = field_map.get(canonical_name)
    if explicit:
        if explicit not in record:
            raise KeyError(
                f"FIELD_MAP maps '{canonical_name}' to '{explicit}', "
                f"but that key is missing. Available keys: {list(record.keys())}"
            )
        return explicit

    for candidate in FIELD_ALIASES[canonical_name]:
        if candidate in record:
            return candidate
    return None


def _iter_input_files(source: str | Path, recursive: bool = True) -> list[Path]:
    source = Path(source)

    if source.is_file():
        if source.suffix.lower() not in SUPPORTED_EXTENSIONS:
            raise ValueError(
                f"Unsupported input type: {source.suffix}. "
                f"Supported: {sorted(SUPPORTED_EXTENSIONS)}"
            )
        return [source]

    if not source.is_dir():
        raise FileNotFoundError(f"Input file/directory not found: {source}")

    iterator = source.rglob("*") if recursive else source.glob("*")
    files = sorted(
        p for p in iterator
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    if not files:
        raise FileNotFoundError(
            f"No supported data files found under {source}. "
            f"Supported: {sorted(SUPPORTED_EXTENSIONS)}"
        )
    return files


def _load_records_from_file(path: Path) -> list[dict]:
    """
    Load either:
      - JSON array: [{...}, {...}]
      - JSON object containing a common list key such as data/documents/chunks/records
      - JSONL / NDJSON / line-delimited JSON
    """
    suffix = path.suffix.lower()

    if suffix == ".json":
        with path.open("r", encoding="utf-8") as fh:
            obj = json.load(fh)

        if isinstance(obj, list):
            records = obj
        elif isinstance(obj, dict):
            records = None
            for key in ("data", "documents", "docs", "chunks", "records", "items"):
                if isinstance(obj.get(key), list):
                    records = obj[key]
                    break
            if records is None:
                # Treat one JSON object as one record.
                records = [obj]
        else:
            raise ValueError(f"Unsupported JSON structure in {path}")

        if not all(isinstance(x, dict) for x in records):
            raise ValueError(f"All records must be JSON objects in {path}")
        return records

    # .jsonl / .ndjson / .txt -> one JSON object per non-empty line.
    records = []
    with path.open("r", encoding="utf-8") as fh:
        for line_number, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON at {path}:{line_number}: {exc}"
                ) from exc
            if not isinstance(record, dict):
                raise ValueError(
                    f"Expected a JSON object at {path}:{line_number}, "
                    f"got {type(record).__name__}"
                )
            records.append(record)
    return records


def _canonicalize_record(
    record: dict,
    generated_id: str,
    field_map: dict,
    source_file: str,
    source_index: int,
) -> dict:
    """Convert a source record to CorpusHop's internal canonical schema."""
    id_key = _pick_field(record, "id", field_map)
    text_key = _pick_field(record, "text", field_map)
    triple_key = _pick_field(record, "triples", field_map)
    entities_key = _pick_field(record, "entities", field_map)
    topic_key = _pick_field(record, "topic", field_map)

    if text_key is None:
        raise KeyError(
            f"Could not find a text field in {source_file} record #{source_index}. "
            f"Set FIELD_MAP['text']. Available keys: {list(record.keys())}"
        )
    if triple_key is None:
        raise KeyError(
            f"Could not find a triple field in {source_file} record #{source_index}. "
            f"Set FIELD_MAP['triples']. Available keys: {list(record.keys())}"
        )

    raw_id = record.get(id_key) if id_key else generated_id
    chunk_id = str(raw_id).strip() if raw_id is not None else generated_id
    if not chunk_id:
        chunk_id = generated_id

    text = record.get(text_key, "")
    if not isinstance(text, str):
        text = str(text)

    triples = record.get(triple_key, [])
    if triples is None:
        triples = []
    if not isinstance(triples, list):
        raise TypeError(
            f"Triple field must be a list in {source_file} record #{source_index}"
        )

    entities = record.get(entities_key, []) if entities_key else []
    if entities is None:
        entities = []
    if not isinstance(entities, list):
        entities = [str(entities)]

    topic = record.get(topic_key, "") if topic_key else ""
    if topic is None:
        topic = ""

    return {
        "id": chunk_id,
        "topic": str(topic),
        "text": text,
        "extracts": triples,
        "entities": [str(e) for e in entities if str(e).strip()],
        "_source_file": source_file,
        "_source_index": source_index,
    }


def load_canonical_documents(
    source: str | Path,
    field_map: dict | None = None,
    recursive: bool = True,
) -> list[dict]:
    """
    Load one file or a directory of files and normalize records to:
      id, topic, text, extracts, entities

    IDs are generated when absent.
    """
    field_map = field_map or {}
    files = _iter_input_files(source, recursive=recursive)

    documents = []
    seen_ids = set()
    global_index = 0

    print(f"Found {len(files)} input file(s).")

    for file_path in files:
        records = _load_records_from_file(file_path)
        print(f"  {file_path}: {len(records):,} record(s)")

        for local_index, record in enumerate(records, 1):
            global_index += 1
            generated_id = f"chunk_{global_index:08d}"
            doc = _canonicalize_record(
                record=record,
                generated_id=generated_id,
                field_map=field_map,
                source_file=str(file_path),
                source_index=local_index,
            )

            if doc["id"] in seen_ids:
                raise ValueError(
                    f"Duplicate chunk id '{doc['id']}'. "
                    "IDs must be unique across all input files."
                )
            seen_ids.add(doc["id"])
            documents.append(doc)

    print(f"Loaded {len(documents):,} total chunk(s).")
    return documents


def _norm(s: str) -> str:
    """
    Unicode-aware exact-match normalization.

    Matching remains exact after normalization:
      - Unicode NFKC normalization
      - case-folding
      - punctuation/symbols -> spaces
      - whitespace collapse

    Letters and digits from any language are retained.
    """
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    chars = []
    for ch in s:
        category = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        elif ch.isspace() or category[0] in {"P", "S"}:
            chars.append(" ")
        else:
            chars.append(" ")
    return " ".join("".join(chars).split())


## 2. Clean triples and keep eligible chunks

In [ ]:
def is_valid_triplet(triplet) -> bool:
    """Return True only for a well-formed [subject, relation, object] triple."""
    if not isinstance(triplet, (list, tuple)) or len(triplet) != 3:
        return False

    subject, relation, obj = triplet
    if not all(isinstance(x, str) and x.strip() for x in (subject, relation, obj)):
        return False

    # Drop self-loops.
    if subject.strip() == obj.strip():
        return False

    return True


def clean_dataset(
    input_source: str | Path,
    output_path: str | Path,
    field_map: dict | None = None,
    recursive: bool = True,
) -> list[dict]:
    """
    Load arbitrary supported input, normalize its schema, clean triples,
    derive entities from surviving triples, and write canonical JSON.
    """
    output_path = Path(output_path)
    raw_docs = load_canonical_documents(
        input_source,
        field_map=field_map,
        recursive=recursive,
    )

    stats = {
        "original_docs": len(raw_docs),
        "docs_after_cleaning": 0,
        "original_total_triplets": 0,
        "cleaned_total_triplets": 0,
        "removed_triplets": 0,
    }

    cleaned_data = []

    for doc in raw_docs:
        original_triplets = doc.get("extracts", [])
        stats["original_total_triplets"] += len(original_triplets)

        valid_triplets = []
        for triplet in original_triplets:
            if is_valid_triplet(triplet):
                valid_triplets.append([str(x).strip() for x in triplet])
            else:
                stats["removed_triplets"] += 1

        # Eligible chunks must contain at least one valid triple.
        if not valid_triplets:
            continue

        # Rebuild entities from the cleaned triples so an input `entities`
        # field is optional.
        valid_entities = sorted(
            {t[0] for t in valid_triplets} | {t[2] for t in valid_triplets}
        )

        cleaned_doc = dict(doc)
        cleaned_doc["extracts"] = valid_triplets
        cleaned_doc["entities"] = valid_entities
        cleaned_data.append(cleaned_doc)

        stats["cleaned_total_triplets"] += len(valid_triplets)

    stats["docs_after_cleaning"] = len(cleaned_data)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as fh:
        json.dump(cleaned_data, fh, ensure_ascii=False, indent=2)

    print("\nData Cleaning Statistics")
    print("=" * 52)
    print(f"Input                  : {input_source}")
    print(f"Output                 : {output_path}")
    print("-" * 52)
    print(f"Original documents     : {stats['original_docs']:,}")
    print(f"Documents after clean  : {stats['docs_after_cleaning']:,}")
    print(f"Original total triples : {stats['original_total_triplets']:,}")
    print(f"Removed invalid triples: {stats['removed_triplets']:,}")
    print(f"Cleaned total triples  : {stats['cleaned_total_triplets']:,}")
    print("=" * 52)

    return cleaned_data


## 3. Build exact-match chunk graph

In [ ]:
import pickle
import time
from collections import defaultdict

import networkx as nx


def create_graph_and_nodes(
    cleaned_file: str | Path,
    output_folder: str | Path,
    graph_filename: str = GRAPH_FILENAME,
    connections_filename: str = CONNECTIONS_FILENAME,
):
    """
    Build the directed chunk graph.

    Edge A -> B exists when a normalized object from a triple in A exactly
    equals a normalized subject from a triple in B.

    No embeddings, fuzzy matching, or semantic similarity are used.
    """
    cleaned_file = Path(cleaned_file)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    if not cleaned_file.exists():
        raise FileNotFoundError(
            f"Cleaned dataset not found: {cleaned_file}. "
            "Run clean_dataset(...) first."
        )

    with cleaned_file.open("r", encoding="utf-8") as fh:
        data = json.load(fh)

    if not isinstance(data, list):
        raise ValueError("Cleaned dataset must be a JSON array.")

    print(f"Loaded {len(data):,} cleaned documents")
    print("Matching strategy: exact normalized object -> subject\n")

    # norm_subject -> list[(chunk_id, triple)]
    print("Building subject index...")
    subj_index = defaultdict(list)

    for doc in data:
        cid = str(doc["id"])
        for triple in doc.get("extracts", []):
            if len(triple) < 3:
                continue
            ns = _norm(triple[0])
            if ns:
                subj_index[ns].append((cid, triple))

    print(f"  Unique normalized subjects: {len(subj_index):,}")

    G = nx.DiGraph()
    for doc in data:
        cid = str(doc["id"])
        G.add_node(cid, **doc)
    print(f"  Nodes: {G.number_of_nodes():,}")

    print("\nCreating edges...")
    edge_details = {}
    t0 = time.time()

    for doc_idx, src_doc in enumerate(data):
        if doc_idx and doc_idx % 300 == 0:
            elapsed = max(time.time() - t0, 1e-6)
            rate = doc_idx / elapsed
            remaining = len(data) - doc_idx
            eta = remaining / rate if rate else 0
            print(
                f"  {doc_idx:,}/{len(data):,}  "
                f"edges={G.number_of_edges():,}  ETA={eta/60:.1f} min"
            )

        src_cid = str(src_doc["id"])

        for src_triple in src_doc.get("extracts", []):
            if len(src_triple) < 3:
                continue

            obj_norm = _norm(src_triple[2])
            if not obj_norm:
                continue

            for dst_cid, dst_triple in subj_index.get(obj_norm, []):
                if dst_cid == src_cid:
                    continue

                key = (src_cid, dst_cid)
                pair = {
                    "src_triple": src_triple,
                    "dst_triple": dst_triple,
                    "entity": obj_norm,
                }

                if key not in edge_details:
                    edge_details[key] = {
                        "src_id": src_cid,
                        "dst_id": dst_cid,
                        "link_entity": obj_norm,
                        "score": 1.0,
                        "pairs": [pair],
                    }
                    G.add_edge(
                        src_cid,
                        dst_cid,
                        link_entity=obj_norm,
                        score=1.0,
                        num_pairs=1,
                    )
                else:
                    edge_details[key]["pairs"].append(pair)
                    G[src_cid][dst_cid]["num_pairs"] += 1

    elapsed = time.time() - t0
    print(f"\nEdge creation done in {elapsed:.1f}s")
    print(f"  Nodes: {G.number_of_nodes():,}")
    print(f"  Edges: {G.number_of_edges():,}")

    print("\nSample A -> B -> C paths:")
    shown = 0
    for a, b in G.edges():
        for c in G.successors(b):
            if c == a:
                continue
            ea, eb = G[a][b], G[b][c]
            print(
                f"  {a} -['{ea['link_entity']}']-> "
                f"{b} -['{eb['link_entity']}']-> {c}"
            )
            shown += 1
            if shown >= 5:
                break
        if shown >= 5:
            break
    if shown == 0:
        print("  (none)")

    connections_file = output_folder / connections_filename
    with connections_file.open("w", encoding="utf-8") as fh:
        fh.write("Directed Graph — exact normalized match\n")
        fh.write("=" * 60 + "\n")
        fh.write(f"Nodes : {G.number_of_nodes()}\n")
        fh.write(f"Edges : {G.number_of_edges()}\n\n")

        for (src, dst), d in edge_details.items():
            fh.write(
                f"{src} -> {dst} "
                f"[entity='{d['link_entity']}' pairs={len(d['pairs'])}]\n"
            )
            for pair in d["pairs"][:3]:
                s, t = pair["src_triple"], pair["dst_triple"]
                fh.write(f"  src: [{s[0]} | {s[1]} | {s[2]}]\n")
                fh.write(f"  dst: [{t[0]} | {t[1]} | {t[2]}]\n")
                fh.write(f"  matched entity: '{pair['entity']}'\n")
            if len(d["pairs"]) > 3:
                fh.write(f"  ... +{len(d['pairs']) - 3} more pairs\n")
            fh.write("\n")

    graph_file = output_folder / graph_filename
    with graph_file.open("wb") as fh:
        pickle.dump(
            {"G": G, "data": data, "edge_details": edge_details},
            fh,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    print(f"\nEdge list: {connections_file}")
    print(f"Graph    : {graph_file}")
    return G, data, edge_details, str(graph_file)


## 4. Greedy 3-hop Bridge selection, question generation, and LLM judge

In [ ]:
"""
multihop_pipeline.py
════════════════════
Unified 3-hop question generation pipeline.

Loop (one path at a time, fully resumable):
  1. Greedy-select the BEST remaining A→B→C path from the live graph.
  2. Generate Q1, Q2, Q3 and the combined multi-hop question via the configured OpenAI model.
  3. Judge the question with an LLM judge.
  4. PASS  → save question; remove all 3 nodes (+ incident edges) from graph.
  5. FAIL  → remove only edges A→B and B→C; try the next best path.
  6. Repeat until no valid path remains or MAX_PATHS is reached.

Degree recalculation
────────────────────
  sub_deg, scores, and order are ALL recomputed fresh from the live graph
  at the start of every call to _pick_best_path, for the touched component.
  This guarantees degree_sum is always accurate after any node/edge removal.

Resume
──────
  Already-judged path keys are loaded from the all-results JSONL on startup;
  those paths (and their nodes/edges) are reconstructed and skipped.
  The graph is rebuilt from the saved removal log so the live graph always
  reflects every decision made so far.

Startup cache
─────────────
  On first run the expensive load+prenorm step runs once, then serialises
  G, edge_details, chunk_to_doc, chunk_index, global_set to
  results/startup_cache/. Subsequent restarts load the cache in ~2-5 min.
  The cache is automatically invalidated whenever source files change.

Component cache
───────────────
  A ComponentCache persists the sparse matrix structure (A_csr, A_csc,
  succ_sets, pred_sets, pi, pj) across iterations. After PASS/FAIL, only
  the touched component is invalidated and rebuilt.

  sub_deg, scores, and order are NOT cached — they are recomputed fresh
  each call from the live graph so they always reflect current degrees.

Outputs  (all in <output_folder>/questions/)
──────────────────────────────────────────────
  multihop_questions_filtered.jsonl
  multihop_questions_filtered_readable.txt
  multihop_questions_all_results.jsonl
  multihop_questions_all_results_readable.txt

Usage
─────
    python multihop_pipeline.py
"""

import hashlib
import json
import os
import pickle
import re
import sqlite3
import time
import traceback
import unicodedata
from datetime import datetime

import networkx as nx
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from scipy import sparse as sp

load_dotenv()

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

GEN_MODEL    = GENERATION_MODEL
JUDGE_MODEL  = JUDGE_MODEL_NAME
TEMPERATURE  = GENERATION_TEMPERATURE
MAX_PATHS    = MAX_SELECTED_PATHS

DB_BATCH_SIZE           = CHAIN_DB_BATCH_SIZE
PHASE2_SAVE_EVERY_RANKS = PHASE2_SAVE_EVERY

OUTPUT_FOLDER = str(OUTPUT_DIR)


# ─────────────────────────────────────────────────────────────────────────────
# Shared utilities
# ─────────────────────────────────────────────────────────────────────────────

def _norm(s: str) -> str:
    """Unicode-aware exact normalization shared with graph construction."""
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    chars = []
    for ch in s:
        category = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        elif ch.isspace() or category[0] in {"P", "S"}:
            chars.append(" ")
        else:
            chars.append(" ")
    return " ".join("".join(chars).split())


def _words(e: str) -> list[str]:
    return re.findall(r"\w+", e.lower())


def _clean_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def _entities_differ(e1: str, e2: str, threshold: float = 0.95) -> bool:
    if e1 == e2:
        return False
    n1 = re.sub(r"[^\w]", "", e1.lower())
    n2 = re.sub(r"[^\w]", "", e2.lower())
    if n1 == n2 or n1 in n2 or n2 in n1:
        return False
    w1, w2 = _words(e1), _words(e2)
    common  = set(w1) & set(w2)
    mn      = min(len(w1), len(w2))
    if mn > 0 and len(common) / mn >= threshold:
        return False
    return True


def _chain_has_duplicate_entities(chain: dict) -> bool:
    t1 = chain["triple_1"]
    t2 = chain["triple_2"]
    t3 = chain["triple_3"]
    entities = [t1[0], t1[2], t2[2], t3[2]]
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            if not _entities_differ(entities[i], entities[j]):
                return True
    return False


def _fmt(triple: list) -> str:
    if not triple or len(triple) < 3:
        return ""
    return f'["{triple[0]}", "{triple[1]}", "{triple[2]}"]'


def _parse_json(raw: str) -> dict:
    clean = (
        raw.strip()
           .removeprefix("```json")
           .removeprefix("```")
           .removesuffix("```")
           .strip()
    )
    return json.loads(clean)


# ─────────────────────────────────────────────────────────────────────────────
# Startup cache
# ─────────────────────────────────────────────────────────────────────────────

def _cache_path(output_folder: str) -> str:
    results_folder = os.path.join(output_folder, "results")
    os.makedirs(results_folder, exist_ok=True)
    return os.path.join(results_folder, "startup_cache.pkl")


def _cache_dir(output_folder: str) -> str:
    results_folder = os.path.join(output_folder, "results")
    cache_dir = os.path.join(results_folder, "startup_cache")
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir


_CACHE_KEYS = ("G", "edge_details", "chunk_to_doc", "chunk_index", "global_set")


def _source_fingerprint(output_folder: str) -> str:
    graph_file   = os.path.join(output_folder, GRAPH_FILENAME)
    cleaned_file = os.path.join(output_folder, CLEANED_FILENAME)
    h = hashlib.md5()
    for f in [graph_file, cleaned_file]:
        if os.path.exists(f):
            st = os.stat(f)
            h.update(f"{f}:{st.st_size}".encode())
            with open(f, "rb") as fh:
                head = fh.read(1_048_576)
                h.update(head)
                if st.st_size > 1_048_576:
                    fh.seek(-1_048_576, os.SEEK_END)
                    h.update(fh.read(1_048_576))
    return h.hexdigest()


def _load_startup_cache(output_folder: str):
    cache_dir   = _cache_dir(output_folder)
    fp_file     = os.path.join(cache_dir, "fingerprint.txt")
    shard_files = {k: os.path.join(cache_dir, f"{k}.pkl") for k in _CACHE_KEYS}
    expected_fp = _source_fingerprint(output_folder)

    if os.path.exists(fp_file) and all(os.path.exists(p) for p in shard_files.values()):
        try:
            with open(fp_file, "r") as fh:
                stored_fp = fh.read().strip()
            if stored_fp != expected_fp:
                print("  Cache stale (source files changed) — rebuilding.")
                return None

            t0 = time.perf_counter()
            print(f"Loading startup cache (split) from {cache_dir} ...")

            from concurrent.futures import ThreadPoolExecutor

            def _load_shard(key):
                with open(shard_files[key], "rb") as fh:
                    return key, pickle.load(fh)

            results: dict = {}
            with ThreadPoolExecutor(max_workers=len(_CACHE_KEYS)) as ex:
                for key, value in ex.map(_load_shard, _CACHE_KEYS):
                    results[key] = value

            elapsed = time.perf_counter() - t0
            G = results["G"]
            print(
                f"  Cache hit in {elapsed:.1f}s — "
                f"{G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges"
            )
            return (
                G,
                results["edge_details"],
                results["chunk_to_doc"],
                results["chunk_index"],
                results["global_set"],
            )
        except Exception as e:
            print(f"  Split cache load failed ({e}) — rebuilding.")
            return None

    legacy_file = _cache_path(output_folder)
    if not os.path.exists(legacy_file):
        return None
    try:
        t0 = time.perf_counter()
        print(f"Loading startup cache (legacy) from {legacy_file} ...")
        with open(legacy_file, "rb") as fh:
            cached = pickle.load(fh)
        if cached.get("fingerprint") != expected_fp:
            print("  Cache stale (source files changed) — rebuilding.")
            return None
        elapsed = time.perf_counter() - t0
        G = cached["G"]
        print(
            f"  Cache hit in {elapsed:.1f}s — "
            f"{G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges"
        )
        result = (
            G,
            cached["edge_details"],
            cached["chunk_to_doc"],
            cached["chunk_index"],
            cached["global_set"],
        )
        try:
            _save_startup_cache(output_folder, *result)
            os.remove(legacy_file)
            print("  Migrated legacy cache to split layout.")
        except Exception as e:
            print(f"  (Could not migrate legacy cache: {e})")
        return result
    except Exception as e:
        print(f"  Cache load failed ({e}) — rebuilding.")
        return None


def _save_startup_cache(
    output_folder: str,
    G,
    edge_details: dict,
    chunk_to_doc: dict,
    chunk_index:  dict,
    global_set:   set,
) -> None:
    cache_dir = _cache_dir(output_folder)
    print(f"Saving startup cache → {cache_dir} ...")

    values = {
        "G":            G,
        "edge_details": edge_details,
        "chunk_to_doc": chunk_to_doc,
        "chunk_index":  chunk_index,
        "global_set":   global_set,
    }

    total_bytes = 0
    for key in _CACHE_KEYS:
        shard_path = os.path.join(cache_dir, f"{key}.pkl")
        tmp = shard_path + ".tmp"
        with open(tmp, "wb") as fh:
            pickle.dump(values[key], fh, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, shard_path)
        total_bytes += os.path.getsize(shard_path)

    fp_file = os.path.join(cache_dir, "fingerprint.txt")
    tmp_fp  = fp_file + ".tmp"
    with open(tmp_fp, "w") as fh:
        fh.write(_source_fingerprint(output_folder))
    os.replace(tmp_fp, fp_file)

    size_mb = total_bytes / 1_048_576
    print(f"  Startup cache saved ({size_mb:.0f} MB across {len(_CACHE_KEYS)} shards).")


# ─────────────────────────────────────────────────────────────────────────────
# Graph / data loading
# ─────────────────────────────────────────────────────────────────────────────

def _load_graph_and_data(output_folder: str) -> tuple:
    graph_file = os.path.join(output_folder, GRAPH_FILENAME)
    if not os.path.exists(graph_file):
        raise FileNotFoundError(graph_file)
    print(f"Loading graph from {graph_file} ...")
    t0 = time.perf_counter()
    with open(graph_file, "rb") as fh:
        gd = pickle.load(fh)
    G            = gd["G"]
    edge_details = gd["edge_details"]
    data         = gd["data"]
    chunk_to_doc = (
        {doc["id"]: doc for doc in data}
        if isinstance(data, list) else data
    )
    elapsed = time.perf_counter() - t0
    print(
        f"  {G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges  "
        f"({elapsed:.1f}s)"
    )
    return G, edge_details, chunk_to_doc


def _load_triple_index(output_folder: str) -> tuple[dict, set]:
    cleaned_file = os.path.join(output_folder, CLEANED_FILENAME)
    if not os.path.exists(cleaned_file):
        return {}, set()
    print(f"Loading triple index ...")
    t0 = time.perf_counter()
    with open(cleaned_file, "r", encoding="utf-8") as fh:
        data = json.load(fh)
    chunk_index: dict[str, frozenset] = {}
    global_set: set[tuple] = set()
    for doc in data:
        cid  = doc["id"]
        keys = set()
        for t in doc.get("extracts", []):
            if len(t) >= 3:
                keys.add((_norm(str(t[0])), _norm(str(t[1])), _norm(str(t[2]))))
        chunk_index[cid] = frozenset(keys)
        global_set.update(keys)
    elapsed = time.perf_counter() - t0
    print(
        f"  {len(chunk_index):,} chunks  |  {len(global_set):,} unique triples  "
        f"({elapsed:.1f}s)"
    )
    return chunk_index, global_set


# ─────────────────────────────────────────────────────────────────────────────
# Pre-normalise edge pairs  (Phase 0)
# ─────────────────────────────────────────────────────────────────────────────

def _prenorm_pairs(edge_details: dict) -> None:
    print("Pre-normalising edge pairs ...")
    t0    = time.perf_counter()
    total = len(edge_details)
    TICK  = max(1, total // 20)
    for i, details in enumerate(edge_details.values()):
        if i % TICK == 0:
            print(f"\r  {100*i//total:3d}%  {i:,}/{total:,}", end="", flush=True)
        for pair in details.get("pairs", []):
            for key in ("src_triple", "dst_triple"):
                norm_key = key + "_norm"
                if norm_key not in pair:
                    raw = pair.get(key)
                    if raw and len(raw) >= 3:
                        pair[norm_key] = (
                            _norm(str(raw[0])),
                            _norm(str(raw[1])),
                            _norm(str(raw[2])),
                        )
                    else:
                        pair[norm_key] = None
    elapsed = time.perf_counter() - t0
    print(f"\r  100%  {total:,}/{total:,}  ({elapsed:.1f}s)")


# ─────────────────────────────────────────────────────────────────────────────
# Chain DB  (Phase 1 — build once, reuse forever)
# ─────────────────────────────────────────────────────────────────────────────

# Bump this whenever the chain-construction semantics change.
CHAIN_LOGIC_VERSION = "bridge_v2_exact_t2_dst_t3"

def _open_chain_db(db_path: str) -> sqlite3.Connection:
    con = sqlite3.connect(db_path, check_same_thread=False)
    con.execute("PRAGMA journal_mode=WAL")
    con.execute("PRAGMA synchronous=NORMAL")
    con.execute("PRAGMA cache_size=-65536")
    con.execute("PRAGMA temp_store=MEMORY")
    con.execute("""
        CREATE TABLE IF NOT EXISTS chains (
            node_a  TEXT NOT NULL,
            node_b  TEXT NOT NULL,
            node_c  TEXT NOT NULL,
            x       TEXT,
            x_norm  TEXT,
            t1_s TEXT, t1_r TEXT, t1_o TEXT,
            t2_s TEXT, t2_r TEXT, t2_o TEXT,
            t3_s TEXT, t3_r TEXT, t3_o TEXT,
            PRIMARY KEY (node_a, node_b, node_c, t1_s, t1_o, t3_o)
        )
    """)
    con.execute("CREATE TABLE IF NOT EXISTS progress (key TEXT PRIMARY KEY, value TEXT)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_ac ON chains (node_a, node_c)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_abc ON chains (node_a, node_b, node_c)")
    con.commit()
    return con


def _db_get(con, key):
    row = con.execute("SELECT value FROM progress WHERE key=?", (key,)).fetchone()
    return row[0] if row else None


def _db_set(con, key, value):
    con.execute("INSERT OR REPLACE INTO progress VALUES (?,?)", (key, value))
    con.commit()


def _build_chain_db(
    edge_details: dict,
    global_set:   set,
    chunk_index:  dict,
    db_path:      str,
) -> sqlite3.Connection:
    validate = bool(chunk_index)
    con      = _open_chain_db(db_path)

    # Never reuse a chain cache produced by an older chain-construction rule.
    stored_version = _db_get(con, "chain_logic_version")
    if stored_version != CHAIN_LOGIC_VERSION:
        con.execute("DELETE FROM chains")
        con.execute("DELETE FROM progress")
        con.commit()
        _db_set(con, "chain_logic_version", CHAIN_LOGIC_VERSION)

    saved_idx    = _db_get(con, "last_bridge_idx")
    resume_from  = (int(saved_idx) + 1) if saved_idx is not None else 0
    total_written = int(_db_get(con, "total_written") or "0")

    print("  Building edge maps ...")
    in_edges:  dict[str, list] = {}
    out_edges: dict[str, list] = {}
    for (src, dst), details in edge_details.items():
        pairs = details.get("pairs", [])
        if not pairs:
            continue
        src_set = chunk_index.get(src, frozenset()) if validate else None
        dst_set = chunk_index.get(dst, frozenset()) if validate else None
        in_edges.setdefault(dst, []).append((src, pairs, src_set))
        out_edges.setdefault(src, []).append((dst, pairs, dst_set))

    bridge_nodes = sorted(set(in_edges) & set(out_edges))
    print(f"  {len(bridge_nodes):,} bridge nodes")
    if resume_from:
        print(f"  Resuming from bridge {resume_from:,} (chains so far: {total_written:,})")

    buf: list = []
    t = time.perf_counter()
    TICK = max(1, len(bridge_nodes) // 200)

    for bi, node_b in enumerate(bridge_nodes):
        if bi < resume_from:
            continue
        if bi % TICK == 0 or bi == len(bridge_nodes) - 1:
            elapsed = time.perf_counter() - t
            pct = 100.0 * (bi + 1) / len(bridge_nodes)
            print(f"\r  [chain DB] {pct:5.1f}%  {bi+1:,}/{len(bridge_nodes):,}  "
                  f"written={total_written:,}  {elapsed:.0f}s",
                  end="\n" if bi == len(bridge_nodes) - 1 else "", flush=True)

        bridge_set = chunk_index.get(node_b, frozenset()) if validate else None

        # For a valid Bridge chain A -> B -> C:
        #   A->B pair: src_triple=t1 in A, dst_triple=t2 in B
        #   B->C pair: src_triple=t2 in B, dst_triple=t3 in C
        # The SAME t2 must participate in both edges.
        bc_index: dict[str, dict[tuple[str, str, str], list]] = {}
        for node_c, bc_pairs, dst_set in out_edges[node_b]:
            options: dict[tuple[str, str, str], list] = {}
            for bc_pair in bc_pairs:
                t2_bc_norm = bc_pair.get("src_triple_norm")
                t3_norm    = bc_pair.get("dst_triple_norm")
                if t2_bc_norm is None or t3_norm is None:
                    continue

                if validate and (
                    t2_bc_norm not in global_set
                    or t2_bc_norm not in bridge_set
                    or t3_norm not in global_set
                    or t3_norm not in dst_set
                ):
                    continue

                # B->C must be the tail-to-head link t2.object == t3.subject.
                if t2_bc_norm[2] != t3_norm[0]:
                    continue

                t3_raw = bc_pair.get("dst_triple") or []
                if len(t3_raw) < 3:
                    continue

                entry = (
                    str(t3_raw[0]).strip(),
                    str(t3_raw[1]).strip(),
                    str(t3_raw[2]).strip(),
                )
                options.setdefault(t2_bc_norm, []).append(entry)

            if options:
                bc_index[node_c] = options

        if bc_index:
            for node_a, ab_pairs, src_set in in_edges[node_b]:
                for node_c, bc_options in bc_index.items():
                    if node_c == node_a:
                        continue
                    for ab_pair in ab_pairs:
                        t1_norm = ab_pair.get("src_triple_norm")
                        t2_norm = ab_pair.get("dst_triple_norm")
                        if t1_norm is None or t2_norm is None:
                            continue
                        if validate and (t1_norm not in src_set or t2_norm not in bridge_set):
                            continue
                        # Continue through B only when the exact same t2 triple
                        # is used by both A->B and B->C.
                        x_norm  = t2_norm[2]
                        t3_list = bc_options.get(t2_norm)
                        if not t3_list:
                            continue
                        ts_raw = ab_pair.get("src_triple") or []
                        td_raw = ab_pair.get("dst_triple") or []
                        if len(ts_raw) < 3 or len(td_raw) < 3:
                            continue
                        s1 = str(ts_raw[0]).strip()
                        r1 = str(ts_raw[1]).strip()
                        s2 = str(ts_raw[2]).strip()
                        r2 = str(td_raw[1]).strip()
                        X  = str(td_raw[2]).strip()
                        for x_raw, r3, s3 in t3_list:
                            buf.append((
                                str(node_a), str(node_b), str(node_c), X, x_norm,
                                s1, r1, s2, s2, r2, X, x_raw, r3, s3,
                            ))
                            if len(buf) >= DB_BATCH_SIZE:
                                con.executemany(
                                    "INSERT OR IGNORE INTO chains VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                                    buf)
                                total_written += len(buf)
                                buf.clear()
                                _db_set(con, "last_bridge_idx", str(bi))
                                _db_set(con, "total_written",   str(total_written))
        bc_index.clear()

    if buf:
        con.executemany(
            "INSERT OR IGNORE INTO chains VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", buf)
        total_written += len(buf)

    _db_set(con, "last_bridge_idx", str(len(bridge_nodes) - 1))
    _db_set(con, "total_written",   str(total_written))
    _db_set(con, "chain_logic_version", CHAIN_LOGIC_VERSION)
    _db_set(con, "phase1_complete", "1")
    con.commit()
    print(f"  Chain DB complete: {total_written:,} rows  ({db_path})")
    return con


def _chain_db_ready(db_path: str) -> bool:
    if not os.path.exists(db_path):
        return False
    try:
        con = sqlite3.connect(db_path)
        phase_row = con.execute(
            "SELECT value FROM progress WHERE key='phase1_complete'"
        ).fetchone()
        version_row = con.execute(
            "SELECT value FROM progress WHERE key='chain_logic_version'"
        ).fetchone()
        con.close()
        return (
            phase_row is not None
            and phase_row[0] == "1"
            and version_row is not None
            and version_row[0] == CHAIN_LOGIC_VERSION
        )
    except Exception:
        return False


def _lookup_chains(
    con: sqlite3.Connection,
    node_a,
    node_c,
    bridge_names: list,
) -> dict[str, list[dict]]:
    if not bridge_names:
        return {}
    node_a       = str(node_a)
    node_c       = str(node_c)
    bridge_names = [str(b) for b in bridge_names]
    ph   = ",".join("?" * len(bridge_names))
    rows = con.execute(
        f"SELECT node_b, x, x_norm, "
        f"t1_s,t1_r,t1_o, t2_s,t2_r,t2_o, t3_s,t3_r,t3_o "
        f"FROM chains "
        f"WHERE node_a=? AND node_c=? AND node_b IN ({ph})",
        (node_a, node_c, *bridge_names),
    ).fetchall()
    grouped: dict[str, list[dict]] = {}
    for row in rows:
        grouped.setdefault(row[0], []).append({
            "X":        row[1],  "X_norm":  row[2],
            "triple_1": [row[3],  row[4],  row[5]],
            "triple_2": [row[6],  row[7],  row[8]],
            "triple_3": [row[9],  row[10], row[11]],
        })
    return grouped


# ─────────────────────────────────────────────────────────────────────────────
# Sparse matrix helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_sparse(nodes: list, node_to_idx: dict, subgraph) -> sp.csr_matrix | None:
    rows, cols = [], []
    for u, v in subgraph.edges():
        rows.append(node_to_idx[u])
        cols.append(node_to_idx[v])
    if not rows:
        return None
    n = len(nodes)
    A = sp.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(n, n))
    A.setdiag(0); A.eliminate_zeros()
    return A


def _build_mink_numpy(na, A_csc, A_csr, sub_deg):
    mink: dict[tuple[int, int], np.float32] = {}
    for i in range(na):
        succ = A_csr.indices[A_csr.indptr[i]:A_csr.indptr[i + 1]]
        for k in succ:
            j_set   = A_csr.indices[A_csr.indptr[k]:A_csr.indptr[k + 1]]  # successors of k
            j_valid = j_set[j_set != i]
            if j_valid.size == 0:
                continue
            dk = sub_deg[k]
            for j in j_valid:
                key = (i, int(j))
                cur = mink.get(key)
                if cur is None or dk < cur:
                    mink[key] = dk
    return mink


# ─────────────────────────────────────────────────────────────────────────────
# Component cache
# ─────────────────────────────────────────────────────────────────────────────
#
# WHAT IS CACHED (structural, stable across FAILs within same component):
#   nodes, node_set, node_to_idx, A_csr, A_csc, succ_sets, pred_sets, pi, pj
#
# WHAT IS NOT CACHED (recomputed fresh every call from live graph):
#   sub_deg  — undirected degree changes whenever any edge is removed (FAIL)
#   scores   — depends on sub_deg via MinK
#   order    — argsort of scores
#
# This means degree_sum is always accurate after every PASS or FAIL.

class _CompEntry:
    """Structural data for one weakly-connected component (degree-independent)."""

    __slots__ = (
        "nodes", "node_set", "node_to_idx",
        "A_csr", "A_csc",
        "pi", "pj",
        "succ_sets", "pred_sets",
    )

    def __init__(self, nodes, node_to_idx, A_csr, A_csc,
                 pi, pj, succ_sets, pred_sets):
        self.nodes       = nodes
        self.node_set    = frozenset(nodes)
        self.node_to_idx = node_to_idx
        self.A_csr       = A_csr
        self.A_csc       = A_csc
        self.pi          = pi
        self.pj          = pj
        self.succ_sets   = succ_sets
        self.pred_sets   = pred_sets


def _build_component_entry(G, comp) -> "_CompEntry | None":
    """
    Build a _CompEntry for one weakly-connected component.
    Only structural data is stored — sub_deg / scores / order are NOT
    stored here; they are recomputed fresh in _pick_best_path each call.
    """
    nodes = list(comp)
    n     = len(nodes)
    if n < 3:
        return None

    node_to_idx = {nd: i for i, nd in enumerate(nodes)}
    subgraph    = G.subgraph(comp)

    A = _build_sparse(nodes, node_to_idx, subgraph)
    if A is None:
        return None
    A_csr = A.tocsr()
    A_csc = A.tocsc()

    sub_undir = nx.Graph(subgraph)
    sub_deg_init = np.array(
        [sub_undir.degree(nodes[i]) for i in range(n)], dtype=np.float32)

    A2 = (A_csr @ A_csr).tocsr()
    A2.setdiag(0)
    A2.data[A2.data < 0.5] = 0.0
    A2.eliminate_zeros()
    if A2.nnz == 0:
        return None

    MinK   = _build_mink_numpy(n, A_csc, A_csr, sub_deg_init)
    A2_coo = A2.tocoo()
    pi, pj = A2_coo.row, A2_coo.col
    mask   = pi != pj
    pi, pj = pi[mask], pj[mask]
    if len(pi) == 0:
        return None

    mink_vals = np.array(
        [MinK.get((int(a), int(b)), np.inf) for a, b in zip(pi, pj)],
        dtype=np.float32,
    )
    valid_mask = mink_vals < np.inf
    if not np.any(valid_mask):
        return None
    pi = pi[valid_mask]
    pj = pj[valid_mask]

    succ_sets = [
        set(A_csr.indices[A_csr.indptr[i]:A_csr.indptr[i + 1]].tolist())
        for i in range(n)
    ]
    pred_sets = [
        set(A_csc.indices[A_csc.indptr[j]:A_csc.indptr[j + 1]].tolist())
        for j in range(n)
    ]

    return _CompEntry(
        nodes=nodes, node_to_idx=node_to_idx,
        A_csr=A_csr, A_csc=A_csc,
        pi=pi, pj=pj,
        succ_sets=succ_sets, pred_sets=pred_sets,
    )


class ComponentCache:
    """
    Persistent cache of structural _CompEntry objects across iterations.
    sub_deg / scores / order are NOT cached — recomputed fresh each call.
    """

    def __init__(self, G):
        self._entries: dict[int, _CompEntry] = {}
        self._next_id = 0
        self._rebuild_all(G)

    def _rebuild_all(self, G) -> None:
        self._entries.clear()
        self._next_id = 0
        components = (
            nx.weakly_connected_components(G)
            if G.is_directed() else nx.connected_components(G)
        )
        for comp in components:
            entry = _build_component_entry(G, comp)
            if entry is not None:
                self._entries[self._next_id] = entry
                self._next_id += 1

    def entries(self):
        return self._entries.values()

    def invalidate_touched(self, G, touched_nodes: list[str]) -> None:
        touched_set = set(touched_nodes)

        stale_ids = [
            cid for cid, entry in self._entries.items()
            if entry.node_set & touched_set
        ]
        if not stale_ids:
            return

        surviving_nodes: set = set()
        for cid in stale_ids:
            for nd in self._entries[cid].nodes:
                if G.has_node(nd):
                    surviving_nodes.add(nd)
            del self._entries[cid]

        if not surviving_nodes:
            return

        sub = G.subgraph(surviving_nodes)
        new_components = (
            nx.weakly_connected_components(sub)
            if sub.is_directed() else nx.connected_components(sub)
        )
        for comp in new_components:
            entry = _build_component_entry(G, comp)
            if entry is not None:
                self._entries[self._next_id] = entry
                self._next_id += 1


# ─────────────────────────────────────────────────────────────────────────────
# Greedy: pick ONE best path — degrees recalculated fresh every call
# ─────────────────────────────────────────────────────────────────────────────

def _pick_best_path(
    G,
    edge_details: dict,
    chunk_to_doc: dict,
    db_con: sqlite3.Connection,
    comp_cache: "ComponentCache",
) -> dict | None:
    best_path  = None
    best_score = np.inf

    for entry in comp_cache.entries():
        nodes      = entry.nodes
        A_csr      = entry.A_csr
        A_csc      = entry.A_csc
        pi, pj     = entry.pi, entry.pj
        succ_sets  = entry.succ_sets
        pred_sets  = entry.pred_sets
        n          = len(nodes)

        sub_undir = nx.Graph(G.subgraph(entry.node_set))
        sub_deg   = np.array(
            [sub_undir.degree(nodes[i]) if G.has_node(nodes[i]) else 0
             for i in range(n)],
            dtype=np.float32,
        )

        MinK = _build_mink_numpy(n, A_csc, A_csr, sub_deg)

        mink_vals = np.array(
            [MinK.get((int(a), int(b)), np.inf) for a, b in zip(pi, pj)],
            dtype=np.float32,
        )
        scores = np.full(len(pi), np.inf, dtype=np.float64)
        valid  = mink_vals < np.inf
        if not np.any(valid):
            continue
        scores[valid] = (
            sub_deg[pi[valid]] + mink_vals[valid] + sub_deg[pj[valid]]
        )
        order = np.argsort(scores, kind="stable")

        for rank in order:
            promised_score = scores[rank]
            if promised_score == np.inf:
                break
            if promised_score >= best_score:
                break

            i_sub = int(pi[rank])
            j_sub = int(pj[rank])
            valid_k_set = (succ_sets[i_sub] & pred_sets[j_sub]) - {i_sub, j_sub}
            if not valid_k_set:
                continue

            valid_k_arr = np.array(list(valid_k_set), dtype=np.intp)
            k_order     = valid_k_arr[np.argsort(sub_deg[valid_k_arr], kind="stable")]

            node_a = nodes[i_sub]
            node_c = nodes[j_sub]

            bridge_names     = [nodes[int(k)] for k in k_order]
            chains_by_bridge = _lookup_chains(db_con, node_a, node_c, bridge_names)

            for k_cand in k_order:
                k_cand   = int(k_cand)
                all_rows = chains_by_bridge.get(str(nodes[k_cand]), [])
                if not all_rows:
                    continue

                valid_chain = None
                for cand_chain in all_rows:
                    if not _chain_has_duplicate_entities(cand_chain):
                        valid_chain = cand_chain
                        break
                if valid_chain is None:
                    continue

                actual_score = float(
                    sub_deg[i_sub] + sub_deg[k_cand] + sub_deg[j_sub]
                )

                if actual_score < best_score:
                    best_score = actual_score
                    x_norm     = valid_chain["X_norm"]

                    def _norm_triple(raw):
                        if not raw or len(raw) < 3:
                            return None
                        return tuple(_norm(str(v)) for v in raw[:3])

                    def _recover_exact_pair(pairs, src_target=None, dst_target=None):
                        for p in pairs:
                            src_norm = p.get("src_triple_norm")
                            dst_norm = p.get("dst_triple_norm")
                            if src_target is not None and src_norm != src_target:
                                continue
                            if dst_target is not None and dst_norm != dst_target:
                                continue
                            return p
                        return {}

                    node_b     = nodes[k_cand]
                    ab_details = edge_details.get((node_a, node_b), {})
                    bc_details = edge_details.get((node_b, node_c), {})

                    t1_target = _norm_triple(valid_chain["triple_1"])
                    t2_target = _norm_triple(valid_chain["triple_2"])
                    t3_target = _norm_triple(valid_chain["triple_3"])

                    # Recover the exact edge-pair evidence used by the selected chain.
                    ab_pair = _recover_exact_pair(
                        ab_details.get("pairs", []),
                        src_target=t1_target,
                        dst_target=t2_target,
                    )
                    bc_pair = _recover_exact_pair(
                        bc_details.get("pairs", []),
                        src_target=t2_target,
                        dst_target=t3_target,
                    )

                    def _nd(cid):
                        doc = chunk_to_doc.get(cid, {})
                        return {
                            "id"      : cid,
                            "topic"   : doc.get("topic",    ""),
                            "text"    : doc.get("text",     ""),
                            "entities": doc.get("entities", []),
                            "extracts": doc.get("extracts", []),
                        }

                    best_path = {
                        "path"       : [node_a, node_b, node_c],
                        "degree_sum" : int(actual_score),
                        "degree_a"   : int(sub_deg[i_sub]),
                        "degree_b"   : int(sub_deg[k_cand]),
                        "degree_c"   : int(sub_deg[j_sub]),
                        "edge_ab"    : {
                            "src": node_a, "dst": node_b,
                            "pairs": ab_details.get("pairs", []),
                        },
                        "edge_bc"    : {
                            "src": node_b, "dst": node_c,
                            "pairs": bc_details.get("pairs", []),
                        },
                        "connected_chain": {
                            "triple_1"      : valid_chain["triple_1"],
                            "triple_2"      : valid_chain["triple_2"],
                            "triple_3"      : valid_chain["triple_3"],
                            "X"             : valid_chain["X"],
                            "X_norm"        : x_norm,
                            "chain_verified": True,
                            "ab_pair"       : ab_pair,
                            "bc_pair"       : bc_pair,
                        },
                        "node_a": _nd(node_a),
                        "node_b": _nd(node_b),
                        "node_c": _nd(node_c),
                    }

                break

    return best_path


# ─────────────────────────────────────────────────────────────────────────────
# Graph update helpers
# ─────────────────────────────────────────────────────────────────────────────

def _graph_remove_pass(G, node_a: str, node_b: str, node_c: str) -> int:
    removed = 0
    for node in [node_a, node_b, node_c]:
        if not G.has_node(node):
            continue
        incident = list(G.in_edges(node)) + list(G.out_edges(node))
        for src, dst in incident:
            if G.has_edge(src, dst):
                G.remove_edge(src, dst)
                removed += 1
        G.remove_node(node)
    return removed


def _graph_remove_fail(G, node_a: str, node_b: str, node_c: str) -> int:
    removed = 0
    for src, dst in [(node_a, node_b), (node_b, node_c)]:
        if G.has_edge(src, dst):
            G.remove_edge(src, dst)
            removed += 1
    return removed


# ─────────────────────────────────────────────────────────────────────────────
# Question generation
# ─────────────────────────────────────────────────────────────────────────────

def _generate_q1(client, text: str, triple: list) -> tuple[str | None, str | None]:
    s1, r1, s2 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST have "{s1}".\n'
        f'  - The relation is "{r1}".\n'
        f'  - The correct answer is "{s2}" — do NOT include "{s2}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s2}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s2
    except Exception as e:
        print(f"    [Q1 error] {e}"); return None, None


def _generate_q2(client, text: str, triple: list) -> tuple[str | None, str | None]:
    s2, r2, s3 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST have "{s2}".\n'
        f'  - The relation is "{r2}".\n'
        f'  - The correct answer is "{s3}" — do NOT include "{s3}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s3}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s3
    except Exception as e:
        print(f"    [Q2 error] {e}"); return None, None


def _generate_q3(client, text: str, triple: list) -> tuple[str | None, str | None]:
    s3, r3, s4 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST have "{s3}".\n'
        f'  - The relation is "{r3}".\n'
        f'  - The correct answer is "{s4}" — do NOT include "{s4}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s4}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s4
    except Exception as e:
        print(f"    [Q3 error] {e}")
        return None, None


def _generate_final(
    client,
    q1: str, a1: str,
    q2: str, a2: str,
    q3: str, a3: str,
) -> tuple[str | None, str | None]:
    instruction = (
        "Generate a multi-hop question and answer based on the provided reasoning chain (Step 1, Step 2, Step 3). "
        "Each step represents a hop in the reasoning path, tagged as <1-hop>, <2-hop>, and <3-hop>. "
        "The final answer must be reached only by traversing all three steps in sequence.\n\n"
        "### Instructions:\n"
        "1. **Generate a Multi-Hop Question**: Use the provided reasoning chain to form ONE natural question that requires "
        "combining information from all three hops (<1-hop> → <2-hop> → <3-hop>). "
        "The question must implicitly require all steps to arrive at the final answer.\n"
        "2. **Preserve the Question Pattern from Step 3**: Extract the exact question type and structure from "
        f"<3-hop> query: '{q3}' (e.g., What is, Who, Where, When) and apply it to the multi-hop context. "
        "Do NOT use generic patterns like 'What entity', 'What organization', or 'What is the name of'.\n"
        "3. **Multi-Hop Reasoning Chain Tags**:\n"
        f"   - <1-hop>: Q: {q1} → A: {a1}\n"
        f"   - <2-hop>: Q: {q2} → A: {a2}\n"
        f"   - <3-hop>: Q: {q3} → A: {a3}\n"
        "   - Ensure the question connects all three hops meaningfully without directly naming "
        f"the intermediate answers '{a1}' or '{a2}'.\n"
        "4. **Descriptive Anchoring**: Ground the question using only descriptive context present in the "
        f"<1-hop> question: '{q1}'. Do NOT introduce external knowledge or information not present in the given steps.\n"
        f"5. **Final Answer Constraint**: The answer must be exactly: '{a3}'. Do not paraphrase or alter it.\n\n"
        "### Avoid Generic Patterns:\n"
        "BAD: 'What entity is associated with...'\n"
        "BAD: 'What organization is connected to...'\n"
        "BAD: 'What is the name of...'\n"
        "BAD: 'What individual...'\n\n"
        "### Prefer Specific Patterns from <3-hop>:\n"
        "GOOD: 'What is the [specific concept from <3-hop>]...'\n"
        "GOOD: 'Who [specific action from <3-hop>]...'\n"
        "GOOD: 'Where [specific location query from <3-hop>]...'\n"
        "GOOD: 'When [specific time query from <3-hop>]...'\n\n"
        "### Output Format:\n"
        f'Return JSON only: {{"question": "multi-hop question preserving <3-hop> pattern", "answer": "{a3}"}}'
    )

    prompt = (
        f"Here is the reasoning chain to build from:\n\n"
        f"<1-hop> Q: {q1} → A: {a1}\n"
        f"<2-hop> Q: {q2} → A: {a2}\n"
        f"<3-hop> Q: {q3} → A: {a3}\n\n"
        f"Following the instructions above, create a single natural multi-hop question that:\n"
        f"- Requires traversing all three hops to reach the final answer\n"
        f"- Preserves the question pattern from <3-hop>: '{q3}'\n"
        f"- Does NOT directly mention intermediate answers '{a1}' or '{a2}'\n"
        f"- Uses only descriptive context from the given steps\n"
        f"- Has exactly '{a3}' as the final answer"
    )

    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        q = result.get("question", "")
        return q, a3
    except Exception as e:
        print(f"    [Final error] {e}")
        return None, None


# ─────────────────────────────────────────────────────────────────────────────
# Validators
# ─────────────────────────────────────────────────────────────────────────────

def _validate_q1(q: str, s2: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s2).lower() in ql:
        print(f"    [Q1 invalid] answer '{s2}' leaked"); return False
    return True

def _validate_q2(q: str, s2: str, s3: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s2).lower() not in ql:
        print(f"    [Q2 invalid] bridge '{s2}' not in question"); return False
    if _clean_ws(s3).lower() in ql:
        print(f"    [Q2 invalid] answer '{s3}' leaked"); return False
    return True

def _validate_q3(q: str, s3: str, s4: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s3).lower() not in ql:
        print(f"    [Q3 invalid] bridge '{s3}' not in question"); return False
    if _clean_ws(s4).lower() in ql:
        print(f"    [Q3 invalid] answer '{s4}' leaked"); return False
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Generate: one path → question set
# ─────────────────────────────────────────────────────────────────────────────

def _generate_for_path(client, path_data: dict) -> dict | None:
    a, b, c = path_data["path"]
    node_a  = path_data.get("node_a", {})
    node_b  = path_data.get("node_b", {})
    node_c  = path_data.get("node_c", {})
    chain   = path_data.get("connected_chain", {})

    def _get(key):
        t = chain.get(key) or []
        if len(t) >= 3:
            return [str(x).strip() for x in t[:3]]
        return []

    t1 = _get("triple_1")
    t2 = _get("triple_2")
    t3 = _get("triple_3")

    if not t1 or not t2 or not t3:
        print(f"  SKIP: malformed connected_chain")
        return None

    s1, r1, s2 = t1
    _,  r2, s3 = t2
    _,  r3, s4 = t3

    print(f"  triple_1 : {_fmt(t1)}")
    print(f"  triple_2 : {_fmt(t2)}")
    print(f"  triple_3 : {_fmt(t3)}  (hint; Q3 answer is free)")
    print(f"  s2 (Q1 ans / Q2 bridge) : '{s2}'")
    print(f"  s3 (Q2 ans / Q3 bridge) : '{s3}'")

    if not _entities_differ(s2, s3):
        print(f"  SKIP: s2='{s2}' too similar to s3='{s3}'")
        return None

    text_a = node_a.get("text", "")
    text_b = node_b.get("text", "")
    text_c = node_c.get("text", "")

    q1, a1 = _generate_q1(client, text_a, t1)
    if not _validate_q1(q1, s2):
        print("  SKIP: Q1 failed"); return None
    print(f"  Q1: {q1}\n  A1: {a1}")

    q2, a2 = _generate_q2(client, text_b, t2)
    if not _validate_q2(q2, s2, s3):
        print("  SKIP: Q2 failed"); return None
    print(f"  Q2: {q2}\n  A2: {a2}")

    q3, a3 = _generate_q3(client, text_c, t3)
    if not _validate_q3(q3, s3, s4):
        print("  SKIP: Q3 failed"); return None
    print(f"  Q3: {q3}\n  A3: {a3}")

    final_q, final_a = _generate_final(client, q1, a1, q2, a2, q3, a3)
    print(f"\n  FINAL Q : {final_q}")
    print(f"  FINAL A : {a3}")

    return {
        "path"           : [a, b, c],
        "degree_sum"     : path_data.get("degree_sum"),
        "triple_1"       : t1,
        "triple_2"       : t2,
        "triple_3"       : t3,
        "entity_x"       : s2,
        "entity_y"       : s3,
        "reasoning_steps": [
            {"hop": 1, "triple": t1, "question": q1, "answer": a1,
             "source_chunk": a, "context": text_a},
            {"hop": 2, "triple": t2, "question": q2, "answer": a2,
             "source_chunk": b, "context": text_b},
            {"hop": 3, "triple": t3, "question": q3, "answer": a3,
             "source_chunk": c, "context": text_c},
        ],
        "final_question" : final_q,
        "final_answer"   : a3,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Judge
# ─────────────────────────────────────────────────────────────────────────────

_JUDGE_SYSTEM = """You are a judge for 3-hop multi-hop questions.

The sub-questions Q1/Q2/Q3 are provided as context only — do NOT judge them.
Your only job is to evaluate the FINAL QUESTION on TWO criteria:

1. MULTI-HOP REASONING — the Final Question must require 3 hops to answer correctly.
2. ANSWER CORRECTNESS — A3 must correctly answer the Final Question based on the provided reasoning steps.

Write one sentence of reasoning, then return JSON on the last line:
{"verdict": "pass"} or {"verdict": "fail", "reason": "..."}
"""


def _judge(client, record: dict) -> tuple[str, str]:
    steps = record.get("reasoning_steps") or []
    s0 = steps[0] if len(steps) > 0 else {}
    s1 = steps[1] if len(steps) > 1 else {}
    s2 = steps[2] if len(steps) > 2 else {}

    user_msg = (
        f"Q1 : {s0.get('question', '')}\n"
        f"A1 : {s0.get('answer', '')}\n\n"
        f"Q2 : {s1.get('question', '')}\n"
        f"A2 : {s1.get('answer', '')}\n\n"
        f"Q3 : {s2.get('question', '')}\n"
        f"A3 : {s2.get('answer', '')}\n\n"
        f"Final Q : {record.get('final_question', '')}\n"
        f"Final A : {record.get('final_answer', '')}\n\n"
        f"Judge on Multi-Hop Reasoning and Answer Correctness only.\n"
        f"One sentence of reasoning, then JSON on the final line."
    )

    def _try_judge():
        resp = client.chat.completions.create(
            model=JUDGE_MODEL, temperature=0.0,
            messages=[
                {"role": "system", "content": _JUDGE_SYSTEM},
                {"role": "user",   "content": user_msg},
            ],
        )
        content = resp.choices[0].message.content.strip()
        if not content:
            raise ValueError("Judge returned empty response")

        # Try every line from the bottom up, looking for valid JSON
        for ln in reversed(content.splitlines()):
            ln = ln.strip()
            if not ln:
                continue
            ln = re.sub(r'^```(?:json)?\s*|\s*```$', '', ln).strip()
            if not ln.startswith("{"):
                continue
            try:
                parsed  = json.loads(ln)
                verdict = parsed.get("verdict", "fail").lower()
                reason  = parsed.get("reason", "")
                if verdict not in ("pass", "fail"):
                    return "fail", "invalid verdict"
                return verdict, reason
            except json.JSONDecodeError:
                continue

        # Last resort: scan entire content for a JSON object
        match = re.search(r'\{[^{}]*"verdict"\s*:\s*"(pass|fail)"[^{}]*\}', content)
        if match:
            try:
                parsed  = json.loads(match.group())
                verdict = parsed.get("verdict", "fail").lower()
                reason  = parsed.get("reason", "")
                return verdict, reason
            except json.JSONDecodeError:
                pass

        raise ValueError(f"No valid JSON verdict found in judge response: {content!r}")

    for attempt in range(3):
        try:
            return _try_judge()
        except Exception as e:
            print(f"    [Judge attempt {attempt+1}/3 failed] {e}")
            time.sleep(1)

    return "fail", "judge error: all 3 attempts failed"


# ─────────────────────────────────────────────────────────────────────────────
# Output helpers
# ─────────────────────────────────────────────────────────────────────────────

def _write_readable(fh, record: dict, verdict: str, reason: str = "") -> None:
    path   = record.get("path", [])
    steps  = record.get("reasoning_steps", [])
    fh.write(f"\n[{verdict.upper()}]  {' → '.join(str(p) for p in path)}\n")
    fh.write(f"entity_x : {record.get('entity_x','')}\n")
    fh.write(f"entity_y : {record.get('entity_y','')}\n")
    if reason:
        fh.write(f"reason   : {reason}\n")
    fh.write("\nReasoning steps:\n")
    for step in steps:
        fh.write(f"  Hop {step.get('hop','')} [{step.get('source_chunk','')}]\n")
        fh.write(f"  Q: {step.get('question','')}\n")
        fh.write(f"  A: {step.get('answer','')}\n\n")
    fh.write(f"Final Q : {record.get('final_question','')}\n")
    fh.write(f"Final A : {record.get('final_answer','')}\n")
    fh.write(("=" if verdict == "pass" else "─") * 60 + "\n")
    fh.flush()


def _load_done_keys(all_results_jsonl: str) -> set[tuple]:
    done = set()
    if not os.path.exists(all_results_jsonl):
        return done
    with open(all_results_jsonl, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                key = tuple(rec.get("path", []))
                if key:
                    done.add(key)
            except Exception:
                pass
    return done


# ─────────────────────────────────────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(output_folder: str = OUTPUT_FOLDER) -> list[dict]:
    """
    Full unified pipeline: greedy-select (fresh degrees) → generate → judge → update graph.
    Fully resumable.
    """
    t0 = time.perf_counter()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Add it to your environment or a local .env file."
        )
    client = OpenAI()

    results_folder   = os.path.join(output_folder, "results")
    questions_folder = os.path.join(output_folder, "questions")
    os.makedirs(results_folder,   exist_ok=True)
    os.makedirs(questions_folder, exist_ok=True)

    db_path              = os.path.join(results_folder,   "chain_cache.db")
    filtered_jsonl       = os.path.join(questions_folder, "multihop_questions_filtered.jsonl")
    filtered_readable    = os.path.join(questions_folder, "multihop_questions_filtered_readable.txt")
    all_results_jsonl    = os.path.join(questions_folder, "multihop_questions_all_results.jsonl")
    all_results_readable = os.path.join(questions_folder, "multihop_questions_all_results_readable.txt")

    print("\n=== Startup ===")
    cached = _load_startup_cache(output_folder)
    if cached:
        G, edge_details, chunk_to_doc, chunk_index, global_set = cached
    else:
        G, edge_details, chunk_to_doc = _load_graph_and_data(output_folder)
        chunk_index, global_set       = _load_triple_index(output_folder)
        print("\n=== Phase 0: Pre-normalise ===")
        _prenorm_pairs(edge_details)
        _save_startup_cache(
            output_folder,
            G, edge_details, chunk_to_doc, chunk_index, global_set,
        )

    print("\n=== Phase 1: Chain DB ===")
    if not _chain_db_ready(db_path):
        db_con = _build_chain_db(edge_details, global_set, chunk_index, db_path)
    else:
        db_con     = _open_chain_db(db_path)
        total_rows = db_con.execute("SELECT COUNT(*) FROM chains").fetchone()[0]
        print(f"  Existing DB: {total_rows:,} chains  ({db_path})")

    done_keys = _load_done_keys(all_results_jsonl)
    accepted_records: list[dict] = []

    if done_keys:
        print(f"\n[Resume] {len(done_keys)} paths already judged — replaying graph updates ...")
        removed_total = 0
        with open(all_results_jsonl, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec     = json.loads(line)
                    verdict = rec.get("verdict", "fail")
                    path    = rec.get("path", [])
                    if len(path) == 3:
                        na, nb, nc = path
                        if verdict == "pass":
                            removed_total += _graph_remove_pass(G, na, nb, nc)
                            accepted_records.append(rec)
                        else:
                            removed_total += _graph_remove_fail(G, na, nb, nc)
                except Exception:
                    pass
        print(f"  Graph edges removed during replay: {removed_total}")
        print(f"  Accepted so far: {len(accepted_records)}")

    print("\n=== Building component cache ===")
    t_cache0 = time.perf_counter()
    comp_cache = ComponentCache(G)
    print(f"  {len(list(comp_cache.entries())):,} candidate components cached "
          f"({time.perf_counter() - t_cache0:.1f}s)")

    print("\n=== Phase 2: Generate → Judge loop (fresh degrees each iteration) ===")
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for path, header in [
        (filtered_readable,    f"FILTERED MULTI-HOP QUESTIONS (PASS ONLY)\nStarted: {now}\n{'='*60}\n"),
        (all_results_readable, f"ALL MULTI-HOP QUESTIONS — JUDGE RESULTS\nStarted: {now}\n{'='*60}\n"),
    ]:
        if not os.path.exists(path):
            with open(path, "w", encoding="utf-8") as fh:
                fh.write(header)

    accepted_count = len(accepted_records)
    failed_count   = 0
    gen_failed     = 0
    iteration      = 0

    with open(filtered_jsonl,       "a", encoding="utf-8") as f_pass, \
         open(all_results_jsonl,    "a", encoding="utf-8") as f_all,  \
         open(filtered_readable,    "a", encoding="utf-8") as f_pass_txt, \
         open(all_results_readable, "a", encoding="utf-8") as f_all_txt:

        while True:
            iteration += 1
            if MAX_PATHS and (accepted_count + failed_count) >= MAX_PATHS:
                print(f"\nReached MAX_PATHS={MAX_PATHS} — stopping.")
                break

            print(f"\n{'='*70}")
            print(f"ITERATION {iteration}  |  "
                  f"Accepted: {accepted_count}  "
                  f"Failed: {failed_count}  "
                  f"Gen-failed: {gen_failed}")
            print(f"Graph: {G.number_of_nodes():,} nodes  "
                  f"{G.number_of_edges():,} edges  |  "
                  f"Cached components: {len(list(comp_cache.entries())):,}")
            print(f"{'='*70}")

            path_data = _pick_best_path(G, edge_details, chunk_to_doc, db_con, comp_cache)
            if path_data is None:
                print("\nNo more valid paths in graph — pipeline complete.")
                break

            node_a, node_b, node_c = path_data["path"]
            print(f"\nSelected path: {node_a} → {node_b} → {node_c}  "
                  f"(score={path_data['degree_sum']})")

            try:
                record = _generate_for_path(client, path_data)
            except Exception as e:
                print(f"  Generation error: {e}")
                traceback.print_exc()
                record = None

            if record is None:
                gen_failed += 1
                _graph_remove_fail(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print(f"  ✗ Generation failed — removed edges A→B, B→C")
                continue

            try:
                verdict, reason = _judge(client, record)
            except Exception as e:
                traceback.print_exc()
                verdict, reason = "fail", f"judge exception: {e}"

            print(f"\n  VERDICT: {verdict.upper()}"
                  + (f"  — {reason}" if reason else ""))

            rec_out                 = dict(record)
            rec_out["verdict"]      = verdict
            rec_out["judge_reason"] = reason

            f_all.write(json.dumps(rec_out, ensure_ascii=False) + "\n")
            f_all.flush()
            _write_readable(f_all_txt, record, verdict, reason)

            if verdict == "pass":
                accepted_count  += 1
                accepted_records.append(record)
                f_pass.write(json.dumps(record, ensure_ascii=False) + "\n")
                f_pass.flush()
                _write_readable(f_pass_txt, record, "pass")
                _graph_remove_pass(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print(f"  ✓ PASS — removed nodes {node_a}, {node_b}, {node_c} + incident edges")
            else:
                failed_count += 1
                _graph_remove_fail(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print(f"  ✗ FAIL — removed edges A→B, B→C only")

    db_con.close()

    elapsed = time.perf_counter() - t0
    print(f"\n{'='*70}")
    print(f"PIPELINE COMPLETE")
    print(f"  Iterations     : {iteration - 1}")
    print(f"  Accepted (PASS): {accepted_count}")
    print(f"  Rejected (FAIL): {failed_count}")
    print(f"  Gen-failed     : {gen_failed}")
    print(f"  Time           : {elapsed:.1f}s")
    print(f"\nOutputs:")
    print(f"  Filtered JSONL    : {filtered_jsonl}")
    print(f"  Filtered readable : {filtered_readable}")
    print(f"  All results JSONL : {all_results_jsonl}")
    print(f"  All results txt   : {all_results_readable}")
    print(f"{'='*70}")

    return accepted_records




## 5. Execute the configured stages

In [ ]:
# =============================================================================
# RUN
# =============================================================================

cleaned_path = OUTPUT_DIR / CLEANED_FILENAME

if RUN_CLEANING:
    clean_dataset(
        input_source=INPUT_DATA,
        output_path=cleaned_path,
        field_map=FIELD_MAP,
        recursive=RECURSIVE_INPUT,
    )

if RUN_GRAPH_BUILDING:
    G, data, edge_details, graph_file = create_graph_and_nodes(
        cleaned_file=cleaned_path,
        output_folder=OUTPUT_DIR,
    )

if RUN_LLM_PIPELINE:
    accepted_records = run_pipeline(output_folder=str(OUTPUT_DIR))
else:
    print(
        "\nLLM generation skipped (RUN_LLM_PIPELINE=False). "
        "Set it to True in the Configuration cell when you are ready."
    )
